In [1]:
# Install dependencies
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.7 MB/s eta 0:00:00


In [20]:
# Load env variables
from anthropic import Anthropic
from google.colab import userdata
from IPython.display import Markdown
from anthropic.types import ToolParam
from datetime import datetime, timedelta
from anthropic.types import Message
import json


In [3]:
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')


In [4]:

# Create an API client
client = Anthropic(
    api_key=my_anthropic_key,
)

print("Anthropic client initialized successfully!")


Anthropic client initialized successfully!


In [5]:
# define a system prompt
#system_prompt = """
#You are a creative screenwriter for feature films.
#"""
system_prompt = None

In [32]:
# Utility functions
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)

def add_assistant_message_with_response_content(messages, response):
    assistant_message = {"role": "assistant", "content": response.content}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=[], tools=None):
    params = {
        "model": my_anthropic_model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)

    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [22]:
# Tools and Schemas

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


In [15]:
def get_tool_use_block(response):
    """
    Safely searches through Claude's response and returns the first ToolUseBlock.
    Returns None if no tool was called.
    """
    for block in response.content:
        if block.type == 'tool_use':
            return block

    return None

def get_tool_use_blocks(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    return tool_requests


In [16]:
def assemble_tool_result(tool_request, tool_output):
  tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": json.dumps(tool_output),
    "is_error": False
  }
  return tool_result_block

def assemble_error_tool_result(tool_request, exception):
  tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": f"Error: {exception}",
    "is_error": True
  }
  return tool_result_block


In [25]:
def run_tool(tool_request):
    tool_output = None
    tool_result_block = None
    tool_name = tool_request.name
    tool_input = tool_request.input
    if tool_name == "get_current_datetime":
        tool_output = get_current_datetime(**tool_input)
        tool_result_block = assemble_tool_result(tool_request, tool_output)
    elif tool_name == "add_duration_to_datetime":
        tool_output = add_duration_to_datetime(**tool_input)
        tool_result_block = assemble_tool_result(tool_request, tool_output)
    elif tool_name == "set_reminder":
        tool_output = set_reminder(**tool_input)
        tool_result_block = assemble_tool_result(tool_request, tool_output)
    return tool_result_block


In [18]:
def run_tools(message):
    tool_requests = get_tool_use_blocks(message)
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
          tool_result_block = run_tool(tool_request)
          tool_result_blocks.append(tool_result_block)
        except Exception as e:
          tool_result_block = assemble_error_tool_result(tool_request, e)
          tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


In [34]:
def run_conversation(messages):
    while True:
        response = chat(messages,
                        tools=[
                            get_current_datetime_schema,
                            add_duration_to_datetime_schema,
                            set_reminder_schema
                        ]
        )
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [35]:
messages = []

add_user_message(messages, "Set a remainder for my doctors appointment. It will be 177 days after Jan 1st, 2050.")
response = run_conversation(messages)
response


I'll help you set a reminder for your doctor's appointment. First, let me calculate what date 177 days after January 1st, 2050 will be.
Now I'll set a reminder for your doctor's appointment on that date:
----
Setting the following reminder for 2050-06-27T00:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. The reminder is now scheduled and you'll be notified at that time.


[{'role': 'user',
  'content': 'Set a remainder for my doctors appointment. It will be 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. First, let me calculate what date 177 days after January 1st, 2050 will be.", type='text'),
   ToolUseBlock(id='toolu_017GUgm4G8xwtNMg3krnRVg9', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017GUgm4G8xwtNMg3krnRVg9',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on that date:", type='text'),
   ToolUseBlock(id='toolu_01Lgycmjf9kbpAxqpHhDTXXm', call